# Classical ML Baseline Practice

**Topic:** Scikit-Learn Pipelines &nbsp;·&nbsp; **Difficulty:** Medium

## The data (created in the next cell)

- `X_train` `(3000, 20)`, `y_train` `(3000,)`, binary labels. Only ~6 of the 20 features are informative. The rest are noise or redundant linear combinations.
- About 4% of training labels are flipped (label noise), so don't chase 100% training accuracy.
- `X_test` `(1000, 20)`: predict these. `y_test` is used by the evaluation cell.

## Your task

1. Compute the majority-class baseline accuracy on the training split so you know what "doing nothing" scores.
2. Train any classical model (logistic regression, k-NN, tree, forest, boosting…) and tune it enough to clearly beat the target.
3. Set `predictions` to your 0/1 predictions for `X_test`.

## Evaluation

Test accuracy must reach the target shown in the evaluation cell.


In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=4000, n_features=20, n_informative=6, n_redundant=6,
    n_clusters_per_class=2, class_sep=1.1, flip_y=0.04, weights=[0.6, 0.4],
    random_state=7,
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1000, random_state=7, stratify=y)
print(f"X_train {X_train.shape}, X_test {X_test.shape}, positive rate {y_train.mean():.2f}")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_ACCURACY = 0.85

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
_true = np.asarray(y_test).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {_cls!r:>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your model and try again.")


## Solution

Check the majority baseline on the training split, then let a random forest handle the redundant/noisy features. Bagging plus feature subsampling is a strong default for tabular problems like this.

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

Xtr, Xval, ytr, yval = train_test_split(X_train, y_train, test_size=0.2, random_state=0)
majority = max(y_train.mean(), 1 - y_train.mean())
print(f"majority baseline: {majority:.3f}")

forest = RandomForestClassifier(n_estimators=400, random_state=0, n_jobs=-1)
forest.fit(Xtr, ytr)
print(f"validation accuracy: {forest.score(Xval, yval):.3f}")

forest.fit(X_train, y_train)
predictions = forest.predict(X_test)
```
